# PyTorchのテンソル（tensor）の扱い方

このノートブックでは、PyTorchのテンソルを扱う際に少し分かりにくいメソッドなどについて解説します。

## `torch.arange` 関数（連番のテンソルを作る）

`torch.arange` は、**一定間隔の連番を持つ 1 次元テンソル** を作る関数です。NumPy の `np.arange` とほぼ同じイメージです。

- 書式: `torch.arange(start, end, step)`
  - **`start`**: 開始値（含む）
  - **`end`**: 終了値（**含まない**）
  - **`step`**: 増分（省略時は 1）
- よくある省略形:
  - `torch.arange(n)` → `0, 1, 2, ..., n-1` のテンソル

### 名前の由来（arange って何？）

`arange` は、**array range（配列として返す range）** を縮めた名前だと考えると分かりやすいです。

- `range` は Python の `range()` と同じく「連続した整数の並び」を表現するイメージ
- その「range」を **array（配列）として返す → array range → arange**

NumPy にもまったく同じ名前・役割の `np.arange` があり、PyTorch の `torch.arange` もそれに対応する関数だと思っておけば OK です。

In [7]:
import torch

# 0 以上 5 未満（0,1,2,3,4）
a = torch.arange(5)
print("a = torch.arange(5):", a)

# 1 以上 6 未満（1,2,3,4,5）
b = torch.arange(1, 6)
print("\nb = torch.arange(1, 6):", b)

# 0 以上 10 未満を 2 ずつ（0,2,4,6,8）
c = torch.arange(0, 10, 2)
print("\nc = torch.arange(0, 10, 2):", c)

# 少数ステップも指定できる（ただし浮動小数の誤差には注意）
d = torch.arange(0.0, 1.0, 0.2)
print("\nd = torch.arange(0.0, 1.0, 0.2):", d)

a = torch.arange(5): tensor([0, 1, 2, 3, 4])

b = torch.arange(1, 6): tensor([1, 2, 3, 4, 5])

c = torch.arange(0, 10, 2): tensor([0, 2, 4, 6, 8])

d = torch.arange(0.0, 1.0, 0.2): tensor([0.0000, 0.2000, 0.4000, 0.6000, 0.8000])


## `view` メソッド（テンソルの形を変える）

`view` は、**テンソルの「形（shape）」だけを変えるメソッド**です。

- 書式: `x.view(new_shape)` または `x.view(dim1, dim2, ...)`
- 中身の要素（値）はそのままで、**並び順も変えずに** 見かけの形だけ変えます。
- `-1` を使うと、「ここは自動で計算して埋めてね」という意味になります。

### よくある使い方のイメージ

- バッチサイズ `batch_size` のベクトルを「列ベクトル」にしたいとき:  
  `actions.view(-1, 1)`  
  → これは「`batch_size × 1` の 2 次元テンソル」に変形する、という意味になります。

In [5]:
import torch

# 1 次元テンソル（要素数 6）
x = torch.tensor([0, 1, 2, 3, 4, 5])
print("x:", x)
print("x.shape:", x.shape)

# 2 行 3 列に変形
x_2x3 = x.view(2, 3)
print("\nx_2x3 = x.view(2, 3):")
print(x_2x3)
print("x_2x3.shape:", x_2x3.shape)

# 3 行 2 列に変形
x_3x2 = x.view(3, 2)
print("\nx_3x2 = x.view(3, 2):")
print(x_3x2)
print("x_3x2.shape:", x_3x2.shape)

# -1 を使って、自動計算してもらう例
x_auto = x.view(-1, 3)  # 「行数はおまかせ、列は 3」
print("\nx_auto = x.view(-1, 3):")
print(x_auto)
print("x_auto.shape:", x_auto.shape)

x: tensor([0, 1, 2, 3, 4, 5])
x.shape: torch.Size([6])

x_2x3 = x.view(2, 3):
tensor([[0, 1, 2],
        [3, 4, 5]])
x_2x3.shape: torch.Size([2, 3])

x_3x2 = x.view(3, 2):
tensor([[0, 1],
        [2, 3],
        [4, 5]])
x_3x2.shape: torch.Size([3, 2])

x_auto = x.view(-1, 3):
tensor([[0, 1, 2],
        [3, 4, 5]])
x_auto.shape: torch.Size([2, 3])


## `view` と `reshape` の違い

`view` と `reshape` は、どちらも「**テンソルの形を変える**」ためのメソッドです。多くの場合は、**結果は同じ** になりますが、内部的な動作に違いがあります。

- **`view`**
  - テンソルがメモリ上で **連続（contiguous）** に並んでいる場合にだけ使えます。
  - 連続でない場合に `view` しようとすると、`RuntimeError: view size is not compatible with ...` のようなエラーになります。

- **`reshape`**
  - できるだけ `view` と同じように動こうとしますが、
  - 必要であれば **内部でコピーを作って** 形を変えることがあります。
  - そのため、「`view` ではエラーになる場合でも、`reshape` は通る」ことがあります。

実務的には、

- **「連続かどうかを意識していて、できるだけコピーを避けたい」→ `view` を使う**
- **「そこまで気にせず、とにかく形だけ変えたい」→ `reshape` を使う**

という感覚で覚えておくと分かりやすいです。

In [6]:
import torch

x = torch.arange(12)  # [0, 1, ..., 11]
print("x:", x)
print("x.shape:", x.shape)

# view と reshape は、この程度なら同じ結果
x_view = x.view(3, 4)
x_reshape = x.reshape(3, 4)
print("\nx_view = x.view(3, 4):")
print(x_view)
print("x_reshape = x.reshape(3, 4):")
print(x_reshape)

# メモリが連続でないテンソルを作る（転置など）
y = x_view.t()  # 転置（transpose）すると、非連続になりやすい
print("\ny = x_view.t():")
print(y)
print("y.is_contiguous():", y.is_contiguous())

# 連続でないテンソルに対して view すると、エラーになることがある
try:
    y_view = y.view(2, 6)
    print("\ny_view = y.view(2, 6):")
    print(y_view)
except RuntimeError as e:
    print("\nview でエラーが出る例:")
    print(e)

# reshape は必要に応じてコピーを作って対応してくれる
y_reshape = y.reshape(2, 6)
print("\ny_reshape = y.reshape(2, 6):")
print(y_reshape)

x: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
x.shape: torch.Size([12])

x_view = x.view(3, 4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
x_reshape = x.reshape(3, 4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

y = x_view.t():
tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
y.is_contiguous(): False

view でエラーが出る例:
view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

y_reshape = y.reshape(2, 6):
tensor([[ 0,  4,  8,  1,  5,  9],
        [ 2,  6, 10,  3,  7, 11]])


## PyTorch の `gather` メソッド

`gather` は、「**インデックスのテンソルを使って、元のテンソルから値を抜き出す**」ためのメソッドです。

PyTorch では、次の **2 通りの呼び出し方** ができます。

1. **関数として呼び出すパターン**  
   - 書式: `torch.gather(input, dim, index)`
   - 例: `out = torch.gather(x, dim=1, index=index)`

2. **テンソルのメソッドとして呼び出すパターン**  
   - 書式: `input.gather(dim, index)`
   - 例: `out = x.gather(dim=1, index=index)`

どちらも **結果は同じ** で、`input` と `dim`、`index` をどう指定するかだけ理解しておけば OK です。

- **引数のおさらい**
  - **`input`**: 元のテンソル（メソッド呼び出しの場合は `self` に相当）
  - **`dim`**: どの次元に沿って要素を取り出すか（0 や 1 などの整数）
  - **`index`**: 取り出したい位置を示すテンソル
- **重要なポイント**
  - `index` の形状は、**出力の形状と同じ**になります。
  - `index` は、`input` と同じ形状か、少なくとも `dim` 以外の次元については同じ形状である必要があります。

強化学習では、Q 値テーブル `q_values` から「実際に選んだ行動の Q 値だけを抜き出す」ときなどによく使われます。

In [4]:
import torch

# 入力テンソル（例）: 2 行 3 列
x = torch.tensor([
    [10, 20, 30],
    [40, 50, 60]
])

# dim=1（列方向）に沿って値を取り出したいとする
# 各行ごとに「どの列を取り出すか」を index で指定する
index = torch.tensor([
    [2, 1],  # 1 行目からは列 2 と列 1 を取り出す
    [0, 0]   # 2 行目からは列 0 と列 0 を取り出す
])

# gather によって、index で指定した位置の値を集める
# 1) 関数として呼び出すパターン
out_func = torch.gather(x, dim=1, index=index)

# 2) テンソルのメソッドとして呼び出すパターン
out_method = x.gather(dim=1, index=index)

print("x =")
print(x)
print("\nindex =")
print(index)
print("\nout_func = torch.gather(x, dim=1, index=index) =")
print(out_func)
print("\nout_method = x.gather(dim=1, index=index) =")
print(out_method)

# 強化学習でのイメージ例
# q_values: 各状態ごとの行動価値（バッチサイズ=3, 行動数=4）
q_values = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [0.5, 1.5, 2.5, 3.5],
    [10.0, 20.0, 30.0, 40.0]
])

# 各状態で実際に選んだ行動（例）
actions = torch.tensor([2, 0, 3])  # 0〜3 の整数

# gather を使うために、actions を 2 次元に変形（列ベクトル）
actions_2d = actions.view(-1, 1)  # shape: (batch, 1)

# dim=1（行動の次元）で、選んだ行動の Q 値だけを抜き出す
chosen_q_values = torch.gather(q_values, dim=1, index=actions_2d)

print("\nq_values =")
print(q_values)
print("\nactions =")
print(actions)
print("\nactions_2d =")
print(actions_2d)
print("\nchosen_q_values =")
print(chosen_q_values)  # shape: (batch, 1)

x =
tensor([[10, 20, 30],
        [40, 50, 60]])

index =
tensor([[2, 1],
        [0, 0]])

out_func = torch.gather(x, dim=1, index=index) =
tensor([[30, 20],
        [40, 40]])

out_method = x.gather(dim=1, index=index) =
tensor([[30, 20],
        [40, 40]])

q_values =
tensor([[ 1.0000,  2.0000,  3.0000,  4.0000],
        [ 0.5000,  1.5000,  2.5000,  3.5000],
        [10.0000, 20.0000, 30.0000, 40.0000]])

actions =
tensor([2, 0, 3])

actions_2d =
tensor([[2],
        [0],
        [3]])

chosen_q_values =
tensor([[ 3.0000],
        [ 0.5000],
        [40.0000]])
